In [ ]:
import os
import shutil
from tqdm import tqdm
import pandas as pd

# Paths (update these)
X = "/home/fahimul/Documents/Research/Dataset/VIGOR"   # folder with files
Y = "/home/fahimul/Documents/Research/Dataset/University1651_vigor/test/query_drone"   # folder with subfolders
Z = "/home/fahimul/Documents/Research/Dataset/University1651_vigor/test/query_satellite"   # destination folder
file_found = 0
csv_file = pd.read_csv('/home/fahimul/Documents/Research/Dataset/VIGOR/splits/VIGOR_test.csv')
os.makedirs(Z, exist_ok=True)
all_gnd_csv = csv_file['gnd_images']
all_sat_csv = csv_file['sat_images']
all_gnd_csv

# Get set of folder names in Y
folder_names_Y = {name for name in os.listdir(Y) if os.path.isdir(os.path.join(Y, name))}


for i in tqdm(range(len(all_gnd_csv))):
    ith_gnd = all_gnd_csv[i].split('/')[2][:-5]
    ith_gnd = ith_gnd.replace(',','_')
   

#     if os.path.isfile(file_path):
#         base_name, ext = os.path.splitext(file_name)

        # check if the file name (without extension) is a folder name in Y
    if ith_gnd in folder_names_Y:
        # make subfolder in Z with the same name as the file (without extension)
        file_found+=1
        src_path = os.path.join(X, all_sat_csv[i])
        target_folder = os.path.join(Z, ith_gnd)
        os.makedirs(target_folder, exist_ok=True)

        # copy file into that folder
        shutil.copy(src_path, os.path.join(target_folder, f'{ith_gnd}.png'))
        print(f"Copied no {file_found}: {ith_gnd}")


In [ ]:
# Changing folder names of drone and satellite images in VIGOR. Ex: pm7P0mMQUMsc2LzwKUdkHw_41.858829_-87.652820 -> 1 
import os

def rename_subfolders(folder1, folder2):
    # Get list of subfolders in both
    subfolders1 = sorted([d for d in os.listdir(folder1) if os.path.isdir(os.path.join(folder1, d))])
    subfolders2 = sorted([d for d in os.listdir(folder2) if os.path.isdir(os.path.join(folder2, d))])

    # Ensure both folders have same subfolder names
    if set(subfolders1) != set(subfolders2):
        raise ValueError("The two folders do not have the same subfolder names.")

    # Sort to get consistent ordering
    subfolders = sorted(subfolders1)

    for idx, sub in enumerate(subfolders, start=1):
        new_name = f"{idx}"

        old_path1 = os.path.join(folder1, sub)
        new_path1 = os.path.join(folder1, new_name)

        old_path2 = os.path.join(folder2, sub)
        new_path2 = os.path.join(folder2, new_name)

        print(f"Renaming '{old_path1}' -> '{new_path1}'")
        print(f"Renaming '{old_path2}' -> '{new_path2}'")

        os.rename(old_path1, new_path1)
        os.rename(old_path2, new_path2)

# Example usage:
rename_subfolders("/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/drone", 
                  "/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/satellite")


Renaming '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/drone/-1Gcc9GDVkxZiaYqzP-2uw_41.852155_-87.679027' -> '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/drone/1'
Renaming '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/satellite/-1Gcc9GDVkxZiaYqzP-2uw_41.852155_-87.679027' -> '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/satellite/1'
Renaming '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/drone/-6G-ukCX4NQ7XgKkGeMX_Q_41.856036_-87.662136' -> '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/drone/2'
Renaming '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/satellite/-6G-ukCX4NQ7XgKkGeMX_Q_41.856036_-87.662136' -> '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/satellite/2'
Renaming '/home/fahimul/Documents/Research/Dataset/University1651_vigor/train/drone/-8SBUT2zEI0pwqhNo7W2-g_41.852088_-87.636488' -> '/home/fahim

## Reducing Number of datapoints

In [6]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import haversine_distances
import math

# ---- Load your CSV ----
ct_nm = 'Seattle' # Chicago NewYork SanFrancisco Seattle

# Assume your CSV has columns: latitude, longitude
df = pd.read_csv(f"csv/vigor_{ct_nm}_final.csv")

df.columns = ["latitude", "longitude", "elevation", "elev_alti"]

# Convert to radians for haversine distance
coords = np.radians(df[['latitude', 'longitude']].values)

thres_m = [30]

for thrs in thres_m:
    # ---- Parameters ----
    # Distance threshold in meters
    threshold_meters = thrs  
    # Earth radius in meters
    earth_radius = 6371008  
    # eps must be in radians
    eps = threshold_meters / earth_radius  

    # ---- Run DBSCAN clustering ----
    db = DBSCAN(eps=eps, min_samples=1, metric='haversine').fit(coords)
    df['cluster'] = db.labels_



    # ---- Reduce to one representative per cluster ----
    # Option 1: Keep first point in each cluster
    reduced_df = df.groupby('cluster').first().reset_index()

    # Option 2 (alternative): Keep cluster centroid
    # reduced_df = df.groupby('cluster')[['latitude','longitude', "elevation", "elev_alti"]].mean().reset_index()

    # Save reduced dataset
    reduced_df.to_csv(f"csv/vigor_{ct_nm}_final_short.csv", index=False)

    print(f"Threshold Meter: {thrs}")
    print(f"Original points: {len(df)}")
    print(f"Reduced points: {len(reduced_df)}\n")


Threshold Meter: 30
Original points: 23750
Reduced points: 1657



In [61]:
import pandas as pd
from tqdm import tqdm
import numpy as np

ct_nm = 'Seattle' # Chicago NewYork SanFrancisco Seattle
df_final_short = pd.read_csv(f"csv/vigor_{ct_nm}_final_short.csv", header=None)
df_all_images = pd.read_csv(f"csv/{ct_nm}_all_images.csv")



common_rows = []

for i in tqdm(range(df_all_images.shape[0])):
    nm = df_all_images['img'][i]
    coordinates = nm.split(',')
    lat_i = np.float64(coordinates[1])
    long_i = np.float64(coordinates[2])
    for j in range(df_final_short.shape[0]):
        lat_j = df_final_short.iloc[j, 0]
        long_j = df_final_short.iloc[j, 1]
        
        if(lat_i==lat_j and long_i==long_j):
            # print(lat_i,lat_j,long_i,long_j)
            common_rows.append(i)
            break
        # else:

# df_new = df_all_images.drop(del_rows)
# del_rows = list(set(del_rows))


100%|██████████| 23751/23751 [10:42<00:00, 36.96it/s]


In [62]:
filtered_df = df_all_images.loc[common_rows]
filtered_df.to_csv(f"csv/{ct_nm}_all_images_short.csv", index=False)

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 83,
 84,
 85,
 86,
 87,
 88,
 91,
 93,
 94,
 95,
 96,
 97,
 98,
 100,
 102,
 103,
 104,
 105,
 106,
 109,
 110,
 111,
 112,
 113,
 114,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 125,
 126,
 127,
 128,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 139,
 140,
 141,
 142,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 154,
 155,
 156,
 157,
 158,
 160,
 161,
 164,
 165,
 166,
 167,
 168,
 169,
 171,
 173,
 174,
 176,
 177,
 178,
 180,
 181,
 182,
 184,
 185,
 187,
 190,
 191,
 192,
 193,
 194,
 195,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 209,
 210,
 213,
 216,
 217,
 218,
 219,
 220,
 221,
 222,
 224,
 22